# 산학프로젝트
**제조 공정 품질 불량 예측**

# 07_1_Episode_Detection

- 샷 단위 지표를 불량 에피소드 단위로 다시 집계
- 평가 프로토콜: 04_6이 저장한 5-fold × 3반복 OOF 확률 재사용, 모델 재학습 없음
- 주의: 에피소드가 6개뿐이므로 통계가 아니라 사례 분석으로 읽을 것

## import

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT=Path.cwd().parent if Path.cwd().name=='data' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# 그래프 한글 표시
plt.rcParams['font.family']='Malgun Gothic'
plt.rcParams['axes.unicode_minus']=False

## 경로설정

In [2]:
if Path.cwd().name!='data':
    os.chdir('data')

## 데이터 불러오기

- 04_6이 저장한 **반복별 OOF 확률**에 생산시각을 결합
- 04_6과 같은 `labeled_modeling.csv`를 **인덱스로 맞춰** 로드

In [3]:
oof_proba=pd.read_csv('04_6_AutoEncoder_CV_Validation_oof_proba.csv', index_col=0)
y_all=oof_proba['PassOrFail'].to_numpy()
n_total=len(y_all)
n_defect=y_all.sum()
print(f'{n_total}행, 불량 {n_defect}건 / 기저 불량률 {n_defect/n_total:.4%}')

# 04_6과 같은 데이터를 인덱스로 맞춤
model_data=pd.read_csv('labeled_modeling.csv', index_col=0)

time_stamp=pd.to_datetime(model_data.loc[oof_proba.index, 'TimeStamp'])
part_name=model_data.loc[oof_proba.index, 'PART_NAME']

# 정렬 검증: 복원한 행의 라벨이 OOF 라벨과 같아야 함
label_check=model_data.loc[oof_proba.index, 'PassOrFail'].to_numpy()
assert (label_check==y_all).all(), '행 정렬이 04_6과 다릅니다'
print('행 정렬 확인 완료')

5230행, 불량 60건 / 기저 불량률 1.1472%
행 정렬 확인 완료


## 1. 에피소드 정의

**불량은 흩어져 나오지 않고 짧은 구간에 몰려서 발생**
- 불량 행을 생산시각 순으로 놓고 간격이 4시간을 넘으면 다른 에피소드로 끊음
- 간격 기준 4시간과 8시간 모두 에피소드 6개, 불량이 발생한 생산일 6일과 일치

In [4]:
EPISODE_GAP_HOURS=4

defect_time=time_stamp[y_all==1].sort_values()
episode_id=pd.Series(-1, index=oof_proba.index, dtype=int)
episode_id.loc[defect_time.index]=(defect_time.diff()>pd.Timedelta(hours=EPISODE_GAP_HOURS)).cumsum().to_numpy()

# 간격 기준에 따른 에피소드 수
for gap_hours in [1, 2, 4, 8, 24]:
    n_episode=(defect_time.diff()>pd.Timedelta(hours=gap_hours)).cumsum().nunique()
    print(f'간격 {gap_hours:>2}시간 초과로 끊으면 에피소드 {n_episode}개')

episode_df=pd.DataFrame({'TimeStamp': time_stamp, 'PassOrFail': y_all, 'Episode': episode_id})
episode_summary=(episode_df[episode_df['Episode']>=0]
                 .groupby('Episode')
                 .agg(**{'시작': ('TimeStamp', 'min'), '종료': ('TimeStamp', 'max'), '불량 수': ('PassOrFail', 'size')}))
episode_summary['지속(분)']=((episode_summary['종료']-episode_summary['시작']).dt.total_seconds()/60).round(1)
episode_summary.index=episode_summary.index+1
episode_summary

간격  1시간 초과로 끊으면 에피소드 9개
간격  2시간 초과로 끊으면 에피소드 8개
간격  4시간 초과로 끊으면 에피소드 6개
간격  8시간 초과로 끊으면 에피소드 6개
간격 24시간 초과로 끊으면 에피소드 5개


,시작,종료,불량 수,지속(분)
Episode,,,,
1,2020-10-16 05:21:38,2020-10-16 05:57:20,17,35.7
2,2020-10-22 00:51:52,2020-10-22 07:37:31,15,405.6
3,2020-10-23 01:03:33,2020-10-23 07:21:20,10,377.8
4,2020-10-27 00:56:21,2020-10-27 01:05:41,10,9.3
5,2020-11-03 04:44:17,2020-11-03 04:44:17,1,0.0
6,2020-11-04 05:17:45,2020-11-04 05:33:13,7,15.5


## 2. 확정 운영점에서의 에피소드 감지

- 05에서 확정한 실시간 Threshold 0.163(검사 물량 10%) 적용
- 에피소드 안의 불량 중 하나라도 알람이 뜨면 감지로 판정
- 감지 순번: 에피소드의 몇 번째 불량에서 처음 잡혔는지
- 감지 지연: 첫 불량으로부터 몇 분 뒤인지

In [5]:
FINAL_MODEL='RandomForest Balanced'
OPERATING_THRESHOLD=0.163
repeat_columns=[f'{FINAL_MODEL} | 원본 | rep{r}' for r in [1, 2, 3]]


def episode_detection(selected):
    rows=[]
    for e in sorted(episode_id[episode_id>=0].unique()):
        member_time=time_stamp[episode_id==e].sort_values()
        member_selected=selected.loc[member_time.index].to_numpy()
        detected=member_selected.any()
        first=int(np.argmax(member_selected)) if detected else np.nan
        rows.append({
            '에피소드': e+1,
            '불량 수': len(member_time),
            '검출 불량': int(member_selected.sum()),
            '감지': bool(detected),
            '감지 순번': np.nan if not detected else first+1,
            '감지 지연(분)': np.nan if not detected else (member_time.iloc[first]-member_time.iloc[0]).total_seconds()/60,
        })
    return pd.DataFrame(rows)


operating_rows=[]
for column in repeat_columns:
    selected=pd.Series(oof_proba[column].to_numpy()>=OPERATING_THRESHOLD, index=oof_proba.index)
    detection=episode_detection(selected)
    detection['Repeat']=column.split('|')[-1].strip()
    detection['알람 수']=int(selected.sum())
    operating_rows.append(detection)

operating_detection=pd.concat(operating_rows, ignore_index=True)
print(f"반복당 알람 {operating_detection['알람 수'].mean():.0f}건 / {n_total}건 ({operating_detection['알람 수'].mean()/n_total:.1%})")
operating_detection

반복당 알람 522건 / 5230건 (10.0%)


,에피소드,불량 수,검출 불량,감지,감지 순번,감지 지연(분),Repeat,알람 수
0,1,17,16,True,1.0,0.000000,rep1,521
1,2,15,9,True,2.0,12.383333,rep1,521
2,3,10,5,True,3.0,57.633333,rep1,521
3,4,10,10,True,1.0,0.000000,rep1,521
4,5,1,0,False,NaN,NaN,rep1,521
5,6,7,7,True,1.0,0.000000,rep1,521
6,1,17,17,True,1.0,0.000000,rep2,522
7,2,15,10,True,2.0,12.383333,rep2,522
8,3,10,4,True,3.0,57.633333,rep2,522
9,4,10,10,True,1.0,0.000000,rep2,522


In [6]:
# 반복 3회 평균
operating_summary=operating_detection.groupby('에피소드').agg(**{
    '불량 수': ('불량 수', 'first'),
    '감지 횟수': ('감지', 'sum'),
    '평균 검출 불량': ('검출 불량', 'mean'),
    '평균 감지 순번': ('감지 순번', 'mean'),
    '평균 감지 지연(분)': ('감지 지연(분)', 'mean'),
})
operating_summary=operating_summary.join(episode_summary[['시작', '지속(분)']])
print(f"에피소드 감지율: {operating_detection['감지'].sum()}/{len(operating_detection)} = {operating_detection['감지'].mean():.3f}")
operating_summary.round({'평균 검출 불량': 2, '평균 감지 순번': 2, '평균 감지 지연(분)': 1, '지속(분)': 1})

에피소드 감지율: 15/18 = 0.833


,불량 수,감지 횟수,평균 검출 불량,평균 감지 순번,평균 감지 지연(분),시작,지속(분)
에피소드,,,,,,,
1,17,3,16.33,1.33,5.3,2020-10-16 05:21:38,35.7
2,15,3,10.00,2.00,12.4,2020-10-22 00:51:52,405.6
3,10,3,4.00,3.67,122.5,2020-10-23 01:03:33,377.8
4,10,3,10.00,1.00,0.0,2020-10-27 00:56:21,9.3
5,1,0,0.00,NaN,NaN,2020-11-03 04:44:17,0.0
6,7,3,7.00,1.00,0.0,2020-11-04 05:17:45,15.5


## 3. 검사 물량별 에피소드 감지

- 05의 검사 물량 표와 같은 정의(전체 상위 k%)로 재계산
- 샷 Recall@k는 05와 같은 값이며, 에피소드 기준 지표를 나란히 배치

In [7]:
inspect_ratios=[0.01, 0.03, 0.05, 0.07, 0.1, 0.15, 0.2]
daily_mean=402

ratio_rows=[]
for ratio in inspect_ratios:
    k=int(round(n_total*ratio))
    detected, shot_recall, order_no, delay=[], [], [], []
    for column in repeat_columns:
        proba=oof_proba[column].to_numpy()
        cut=np.sort(proba)[::-1][k-1]
        selected=pd.Series(proba>=cut, index=oof_proba.index)
        detection=episode_detection(selected)
        detected.append(detection['감지'].sum())
        shot_recall.append(detection['검출 불량'].sum()/n_defect)
        order_no.append(detection['감지 순번'].mean())
        delay.append(detection['감지 지연(분)'].mean())
    ratio_rows.append({
        '검사 비율': f'{ratio:.0%}',
        '하루 검사 건수': round(daily_mean*ratio),
        '에피소드 감지': f'{np.mean(detected):.1f}/6',
        '샷 Recall@k': round(np.mean(shot_recall), 3),
        '평균 감지 순번': round(np.nanmean(order_no), 2),
        '평균 감지 지연(분)': round(np.nanmean(delay), 1),
    })

ratio_detection=pd.DataFrame(ratio_rows)
ratio_detection

,검사 비율,하루 검사 건수,에피소드 감지,샷 Recall@k,평균 감지 순번,평균 감지 지연(분)
0,1%,4,3.3/6,0.322,5.11,30.2
1,3%,12,4.7/6,0.450,2.77,43.1
2,5%,20,4.7/6,0.533,2.60,42.9
3,7%,28,5.0/6,0.683,2.27,44.1
4,10%,40,5.0/6,0.789,1.80,28.0
5,15%,60,5.0/6,0.861,1.67,24.7
6,20%,80,5.0/6,0.939,1.20,4.1


## 4. Threshold 고정 시 생산일별 알람

- 운영 방식 결정(`dashboard/plan.md` 7절)의 근거 표. Threshold 0.163을 고정했을 때 알람이 날마다 얼마나 변동하는지 확인
- 04_6 OOF 확률 반복 3회에 각각 적용한 뒤 생산일별로 평균. 알람 · 검출은 반복 3회 평균이라 소수가 나올 수 있음
- 함께 보는 값: 시간당 알람 분포(물량 상한 근거), 일별 확률 90분위(Threshold 재조정 근거)


In [8]:
production_date=time_stamp.dt.date
production_hour=time_stamp.dt.floor('h')

daily_rows, hourly_rows, q90_rows=[], [], []
for column in repeat_columns:
    proba=oof_proba[column].to_numpy()
    frame=pd.DataFrame({'date': production_date.to_numpy(), 'hour': production_hour.to_numpy(),
                        'defect': y_all, 'alarm': (proba>=OPERATING_THRESHOLD).astype(int), 'proba': proba})
    frame['caught']=frame['defect']*frame['alarm']
    daily_rows.append(frame.groupby('date').agg(**{'생산': ('defect', 'size'), '실제 불량': ('defect', 'sum'),
                                                  '알람': ('alarm', 'sum'), '검출': ('caught', 'sum')}))
    hourly_rows.append(frame.groupby('hour')['alarm'].sum())
    q90_rows.append(frame.groupby('date')['proba'].quantile(0.9))

# 반복 3회 평균
daily_alarm=pd.concat(daily_rows).groupby(level=0).agg(**{'생산': ('생산', 'first'), '실제 불량': ('실제 불량', 'first'),
                                                         '알람': ('알람', 'mean'), '검출': ('검출', 'mean')})
daily_alarm['알람률']=daily_alarm['알람']/daily_alarm['생산']
daily_alarm.index=pd.to_datetime(daily_alarm.index).strftime('%m-%d')
daily_alarm.index.name='생산일'

hourly_alarm=pd.concat(hourly_rows, axis=1).mean(axis=1)
defect_hours=int((pd.Series(y_all).groupby(production_hour.to_numpy()).sum()>0).sum())
daily_q90=pd.concat(q90_rows, axis=1).mean(axis=1)

total_alarm=daily_alarm['알람'].sum()
total_caught=daily_alarm['검출'].sum()
print(f'생산일 {len(daily_alarm)}일 합계: 알람 {total_alarm:.0f}건 ({total_alarm/n_total:.1%}), 검출 {total_caught:.1f}/{n_defect}건 ({total_caught/n_defect:.1%})')
print(f'일별 알람: {daily_alarm["알람"].min():.0f} ~ {daily_alarm["알람"].max():.0f}건')
print(f'시간당 알람: 중앙값 {hourly_alarm.median():.0f}건, 90분위 {hourly_alarm.quantile(0.9):.0f}건, 최대 {hourly_alarm.max():.0f}건'
      f' (생산이 있던 {len(hourly_alarm)}시간 중 불량이 난 시간 {defect_hours}시간)')
print(f'일별 확률 90분위: {daily_q90.min():.3f} ~ {daily_q90.max():.3f}')
daily_alarm.round({'알람': 1, '검출': 1, '알람률': 3})


생산일 13일 합계: 알람 522건 (10.0%), 검출 47.3/60건 (78.9%)
일별 알람: 0 ~ 165건
시간당 알람: 중앙값 0건, 90분위 28건, 최대 80건 (생산이 있던 59시간 중 불량이 난 시간 15시간)
일별 확률 90분위: 0.003 ~ 0.688


,생산,실제 불량,알람,검출,알람률
생산일,,,,,
10-16,375,17,115.0,16.3,0.307
10-19,72,0,0.0,0.0,0.000
10-20,764,0,4.0,0.0,0.005
10-21,82,0,9.3,0.0,0.114
10-22,600,15,165.3,10.0,0.276
10-23,500,10,133.3,4.0,0.267
10-27,628,10,17.0,10.0,0.027
10-29,454,0,0.0,0.0,0.000
10-30,755,0,4.0,0.0,0.005


## 5. 정리

**검사 물량 10%(Threshold 0.163)에서 에피소드 6개 중 5개 감지**
- 놓친 1개는 불량이 1건뿐인 단발
- 불량이 2건 이상 이어진 에피소드 5개는 반복 3회 모두 감지
- 에피소드 4·6은 반복 3회 모두 첫 번째 불량 샷에서 포착, 에피소드 1은 3회 중 2회

**Threshold 0.163 고정 시 알람은 하루 0 ~ 165건으로 변동 (반복 3회 평균)**
- 13일 합계 알람 522건(10.0%), 검출 47.3/60건(78.9%)으로 검사 물량 10% 방식과 총량이 같음
- 불량이 몰린 날(10-16 · 10-22 · 10-23)에 알람이 115 ~ 165건으로 몰리고, 불량이 없던 날은 0 ~ 9건
- 10-27은 알람 17건으로 불량 10건을 전부 검출. 물량 방식이면 같은 날 63건을 검사해야 함
- 시간당 알람은 중앙값 0건, 90분위 28건, 최대 80건. 일별 확률 90분위는 0.003 ~ 0.688로 움직임
- 이 표가 `dashboard/plan.md` 7절(실시간 Threshold + 물량 상한 채택)의 근거

**검사 물량 7% 이상에서 에피소드 감지 5개로 포화**
- 물량을 더 늘려 얻는 것은 새 에피소드가 아니라 같은 에피소드 안의 검출 건수

**한계**
- 에피소드 6개뿐이므로 통계가 아니라 사례 분석
- 무작위 CV 확률 기준이라 같은 에피소드의 다른 샷이 학습에 들어간 조건
- 시간순 조건 수치는 07_2에 있음

In [9]:
episode_summary.to_csv('07_1_Episode_Detection_episodes.csv', encoding='utf-8-sig')
operating_summary.to_csv('07_1_Episode_Detection_operating.csv', encoding='utf-8-sig')
ratio_detection.to_csv('07_1_Episode_Detection_by_ratio.csv', index=False, encoding='utf-8-sig')
daily_alarm.to_csv('07_1_Episode_Detection_daily_alarm.csv', encoding='utf-8-sig')
print('저장 완료')

저장 완료
